In [8]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash
import dash_bootstrap_components as dbc
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64


# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bson.json_util import dumps

#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from mongodb_crud import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "<my_password>"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
#print(len(df.to_dict(orient='records')))
#print(df.columns)



#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)
app = dash.Dash(external_stylesheets=[dbc.themes.BOOTSTRAP])

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    #html.Div(id='hidden-div', style={'display':'none'}),
    # HTML image tag
    html.Div(
            html.A(
                href="https://www.snhu.edu",
                    children=[
                        html.Img(
                            src='data:image/png;base64,{}'.format(encoded_image.decode()), 
                            style={
                                'height':'8%',
                                'width':'8%',
                            }
                        ),
                    ],
                    
            ),
        style={'textAlign':'center'}
    ),
            
    html.Center(
        html.B(
            html.H1(
                "SNHU CS-340 Dashboard"
            )
                      
        )
    ),
                  
                          
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
    
    html.Hr(),
    # Interactive filtering options
    html.Div(className='row',
        style={'display' : 'flex'},
        children= [
            dcc.RadioItems(
                id='filter-type',
                options=[
                    {'label': 'Water Rescue','value': 'Water'},
                    {'label': 'Mountain Rescue', 'value': 'Mountain'},
                    {'label': 'Disaster Rescue', 'value': 'Disaster'},
                    {'label': 'Reset', 'value': 'Reset'},
                ],
                inline=True
            ),
        ]  
    ),
             
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                        data=df.to_dict('records'),
                        
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here
                        editable=True,
                        row_selectable="single", # allows a row to be selected
                        selected_rows=[0],
                        selected_columns=[],
                        row_deletable=False,
                        filter_action='native',# allows a filter
                        sort_action='native', #allows sorting
                        sort_mode='multi', #sets the sorting mode to mutiple columns
                        page_action="native", #enables pagination
                        page_current= 0, #sets start page
                        page_size= 10, #sets rows per page
                        ),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
        style={'display' : 'flex'},
        children=[
            html.Div(
                id='graph-id',
                className='col s12 m6',

            ),
            html.Div(
                id='map-id',
                className='col s12 m6',
            )
        ]
    ),
    html.Footer(
        html.Center(html.I("Developed by Dominick Marquez"))
    )
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback([Output('datatable-id','data'),
              Output('datatable-id', 'columns')],
              [Input('filter-type', 'value')])
def update_dashboard(filter):
## FIX ME Add code to filter interactive data table with MongoDB queries
    
    # Water Rescue Filter
    if filter == 'Water':
        query = {
            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },
            "sex_upon_outcome":"Intact Female",
            "$and": [
                {"age_upon_outcome_in_weeks": {"$gte":26}},
                {"age_upon_outcome_in_weeks": {"$lte":156}}
            ]
        }
        
    # Mountain Rescue Filter
    elif filter == 'Mountain':
        query = {
            "breed": {
                "$in": [
                    "German Shepherd", 
                    "Alaskan Malamute", 
                    "Old English Sheepdog", 
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "$and": [
                {"age_upon_outcome_in_weeks": {"$gte": 26}},
                {"age_upon_outcome_in_weeks": {"$lte": 156}}
            ]
        }
        
    # Disaster Rescue Filter
    elif (filter == 'Disaster'):
        query = {
            "breed": {
                "$in": [
                    "Doberman Pinscher", 
                    "German Shepherd", 
                    "Golden Retriever", 
                    "Bloodhound",
                    "Rottweiler"
                ]
            },
            "sex_upon_outcome": "Intact Male",
            "$and" : [
                {"age_upon_outcome_in_weeks": {"$gte": 20}}, 
                {"age_upon_outcome_in_weeks": {"$lte": 300}}
            ]
        }
        
    #Reset Filter
    else:
        query = {}
        
    df = pd.DataFrame(list(db.readAll(query)))
    
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data = df.to_dict('records')
    
    return (data, columns)
    
# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_viewport_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    dff = pd.DataFrame.from_dict(viewData)

    return [
        dcc.Graph(
            style={"width": "1300px", "height": "500px"},
            figure = px.pie(dff, names='breed', color_discrete_sequence=px.colors.sequential.RdBu)
        )    
    ]
    
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
#FIXME Add in the code for your geolocation chart    
    dff = pd.DataFrame.from_dict(viewData)
    # Because we only allow single row selection, the list can 
    # be converted to a row index here
    if index is None:
        row = 0
    else: 
        row = index[0]

    # Austin TX is at [30.75,-97.48]
    return [
        dl.Map(style={'width': '1050px', 'height': '450px'},
            center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for 
            # the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
                dl.Marker(position=[dff.iloc[row,13],dff.iloc[row,14]],
                    children=[
                    dl.Tooltip(dff.iloc[row,4]),
                    dl.Popup([
                        html.H1("Animal Name"),
                        html.P(dff.iloc[row,9])
                    ])
                ])
        ])
    ]


app.run_server(mode='external',host='127.0.0.1', port=30760)


/home/dominickmarqu_snhu/.local/lib/python3.9/site-packages/dash/dash.py:556: UserWarning:

JupyterDash is deprecated, use Dash instead.
See https://dash.plotly.com/dash-in-jupyter for more details.

